# Adaptive Model Selection v2.0 - Execution Notebook

Runs the evidence-based model selection engine (`Functions/`) on the IF quantification dataset (treatment comparison: Ctrl vs. Laser vs. TGFB) at the 7d timepoint. Analyzes each marker (Col, FIB, VIM, aSMA, Ki67), producing one Excel workbook, a set of diagnostic plots, and one HTML report per analysis.

## Step 0: Libraries

In [ ]:
library(openxlsx)   # or `library(xlsx)` - whichever your read.xlsx() comes from
library(dplyr)

## Step 1: Source all modules

`module_dir` is the only path to update if the module folder moves. Files are sourced in dependency order (`helper_functions.R` first, `run_adaptive_model_selection_v2.R` last as the orchestrator).

In [ ]:
base_dir <- getwd()

module_dir <- file.path(base_dir, "Functions")
module_files <- c(
  "helper_functions.R",
  "distribution_assessment.R",
  "model_candidates.R",
  "diagnostics.R",
  "scoring.R",
  "reporting.R",
  "plotting.R",
  "run_adaptive_model_selection_v2.R"
)

for (f in module_files) {
  source(file.path(module_dir, f))
}
report_rmd <- file.path(module_dir, "LmmGmm_adaptive_model_selection_Report_v2_0.Rmd")

## Step 2: Define the analysis helper function

`run_adaptive_analysis()` wraps one full analysis run - filtering to the current marker/timepoint, model selection, and exporting the xlsx/plots/HTML report - so each marker analysis below is a single function call. Raw data is assumed pre-filtered to the relevant layers/treatments upstream; this function only selects the current marker (`target`) and timepoint.

In [ ]:
run_adaptive_analysis <- function(
    dataset,
    target,
    current_timepoint,
    dataset_descriptor,
    results_dir,
    measurement_variables,
    comparison_parameter,
    defined_contrasts = NULL,   # optional: list(c("levelA","levelB"), ...) to restrict pairwise comparisons to specific pairs instead of all-pairs
    comparison_levels = NULL,
    confounder_vars = c("batchNo"),
    correlated_slopes = FALSE,
    alpha = 0.05,
    fit_empirical = TRUE,
    p_adjust_method = "BH"
) {

  # Filter to the current marker and timepoint
  dataset <- dataset |>
    dplyr::filter(
      timepoint == current_timepoint,
      target == !!target
    )

  if (nrow(dataset) == 0) {
    stop("No observations remain after filtering.")
  }

  # Set factor order (optional)
  if (!is.null(comparison_levels)) {
    dataset[[comparison_parameter]] <- factor(
      dataset[[comparison_parameter]],
      levels = comparison_levels
    )
  }

  # Run adaptive model selection
  batch_results <- run_adaptive_model_selection_batch(
    datamatrix            = dataset,
    measurement_variables = measurement_variables,
    comparison_parameter  = comparison_parameter,
    confounder_vars       = confounder_vars,
    correlated_slopes     = correlated_slopes,
    alpha                 = alpha,
    fit_empirical         = fit_empirical,
    p_adjust_method       = p_adjust_method,
    defined_contrasts     = defined_contrasts
  )

  # Create output directory and run tag
  dir.create(results_dir, recursive = TRUE, showWarnings = FALSE)

  run_tag <- paste0(
    dataset_descriptor,
    "_",
    current_timepoint,
    "_conf_",
    paste(confounder_vars, collapse = "_")
  )

  # Export tables as one multi-sheet workbook
  manuscript_table <- build_manuscript_table(batch_results$supplementary_table)
  openxlsx::write.xlsx(
    list(
      Final_summary                 = batch_results$final_summary,
      Supplementary_table           = manuscript_table$table,
      Supplementary_table_footnotes = manuscript_table$footnotes,
      Supplementary_table_full      = batch_results$supplementary_table,
      Sample_size_table             = batch_results$sample_size_table,
      Comparison_table              = batch_results$comparison_table,
      Raw_data_table                = batch_results$raw_data_table,
      Candidate_table               = batch_results$candidate_table,
      Scoring_log                   = batch_results$scoring_log,
      Anova_table                   = batch_results$anova_table,
      Pairwise_table                = batch_results$pairwise_table,
      Random_eff_table              = batch_results$random_eff_table,
      Session_info                  = build_session_info_table()
    ),
    file = file.path(
      results_dir,
      paste0("AdaptiveModelSelection_", run_tag, ".xlsx")
    ),
    overwrite = TRUE
  )

  # Export diagnostic plots
  plot_dir <- file.path(results_dir, "diagnostic_plots")
  dir.create(plot_dir, recursive = TRUE, showWarnings = FALSE)

  for (meas in names(batch_results$diagnostics_plots)) {
    ggplot2::ggsave(
      filename = file.path(plot_dir, paste0(meas, "_", run_tag, ".png")),
      plot = batch_results$diagnostics_plots[[meas]],
      width = 9,
      height = 4,
      dpi = 300
    )
  }

  # Render HTML report
  rmarkdown::render(
    input = report_rmd,
    params = list(batch_results = batch_results),
    output_file = paste0("Spheroids_AdaptiveModelSelection_", run_tag, ".html"),
    output_dir = results_dir,
    envir = new.env(parent = globalenv())
  )

  message("Analysis completed successfully.")
  message("Results saved to: ", normalizePath(results_dir))

  invisible(batch_results)
}

## Step 3: Load raw data

In [ ]:
full_data <- read.xlsx(file.path(base_dir, "Rawdata_Files", "SegmentationData_Treatments_FibrosisMarker.xlsx"))
head(full_data, n = 5)

Ki67_data <- read.xlsx(file.path(base_dir, "Rawdata_Files", "SegmentationData_Treatments_Ki67.xlsx"), sheet = 1)
head(Ki67_data, n = 5)

## Analysis: Collagen (Col)

In [ ]:
results <- run_adaptive_analysis(
  dataset = full_data,
  target = "COL",
  current_timepoint = "7d",
  dataset_descriptor = "COL",
  results_dir = file.path(base_dir, "Report_Files/Treatment_Reports"),
  measurement_variables = c("mean_int"),
  comparison_parameter = "treatment",
  comparison_levels = c("Ctrl", "Laser", "TGFB")
)

## Analysis: Fibronectin (FIB)

In [ ]:
results <- run_adaptive_analysis(
  dataset = full_data,
  target = "FIB",
  current_timepoint = "7d",
  dataset_descriptor = "FIB",
  results_dir = file.path(base_dir, "Report_Files/Treatment_Reports"),
  measurement_variables = c("mean_int"),
  comparison_parameter = "treatment",
  comparison_levels = c("Ctrl", "Laser", "TGFB")
)

## Analysis: Vimentin (VIM)

In [ ]:
results <- run_adaptive_analysis(
  dataset = full_data,
  target = "VIM",
  current_timepoint = "7d",
  dataset_descriptor = "VIM",
  results_dir = file.path(base_dir, "Report_Files/Treatment_Reports"),
  measurement_variables = c("mean_int"),
  comparison_parameter = "treatment",
  comparison_levels = c("Ctrl", "Laser", "TGFB")
)

## Analysis: alpha-SMA (aSMA)

In [ ]:
results <- run_adaptive_analysis(
  dataset = full_data,
  target = "aSMA",
  current_timepoint = "7d",
  dataset_descriptor = "aSMA",
  results_dir = file.path(base_dir, "Report_Files/Treatment_Reports"),
  measurement_variables = c("mean_int"),
  comparison_parameter = "treatment",
  comparison_levels = c("Ctrl", "Laser", "TGFB")
)

## Analysis: Proliferation (Ki67)

In [ ]:
results <- run_adaptive_analysis(
  dataset = Ki67_data,
  target = "Ki67",
  current_timepoint = "7d",
  dataset_descriptor = "Ki67",
  results_dir = file.path(base_dir, "Report_Files/Treatment_Reports"),
  measurement_variables = c("counts_per_Vol"),
  comparison_parameter = "treatment",
  comparison_levels = c("Ctrl", "Laser", "TGFB")
)